In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
from matplotlib.lines import Line2D
sys.path.insert(0,'..')
from scripts.utils import Config,load_stats
from scripts.models.sr.train import load_features
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
CONFIG      = Config()
STATS       = load_stats(CONFIG.splitsdir)
SPLITSDIR   = CONFIG.splitsdir
PREDSDIR    = CONFIG.predsdir
NNRUNS      = CONFIG.nn['runs']
EQUATIONS   = CONFIG.sr['optimizedeqs']
ATMRUN      = CONFIG.sr['runs']['sr_atm']
LABELS      = {**{name:run['description'] for name,run in NNRUNS.items()},**{name:eq['description'] for name,eq in EQUATIONS.items()}}
COLORS      = {'ERA5':'gray','IMERG':'k',**{name:eq['color'] for name,eq in EQUATIONS.items()}}
CYCLEMODELS = ['sr_atm_eq','sr_sfc_eq','sr_all_pc_eq']
PLOTMODELS  = ['sr_atm_eq','sr_all_pc_eq']
SPLIT       = 'test'
TIMEWINDOW  = CONFIG.timewindow
NHOURBINS   = 8
LANDFRAC    = 0.5
MINPRECIP   = 0.1
TOPQUANTILE = 0.8
NBINS       = 20
MINSAMPLES  = 50
FIGPATH     = '../figs/fig_imerg.jpg'

In [ ]:
def load_predictions(name):
    path = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(path):
        return None
    with xr.open_dataset(path) as ds:
        da = ds['tp'].load()
    if 'seed' in da.dims:
        return da.transpose('seed','time','lat','lon').values.reshape(da.sizes['seed'],-1)
    return da.squeeze().transpose('time','lat','lon').values.reshape(1,-1)

def calc_r2(pred,true,mask):
    p,t = pred[mask],true[mask]
    return 1-np.sum((p-t)**2)/np.sum((t-t.mean())**2)

def calc_diurnal_cycle(values,mask):
    counts = np.bincount(hourbin[mask],minlength=NHOURBINS)
    return np.bincount(hourbin[mask],weights=values[mask],minlength=NHOURBINS)/np.maximum(counts,1)

def bin_mean(x,y,values,edges):
    binidx = np.clip(np.digitize(x,edges[0])-1,0,NBINS-1)*NBINS+np.clip(np.digitize(y,edges[1])-1,0,NBINS-1)
    counts = np.bincount(binidx,minlength=NBINS**2)
    return np.where(counts>=MINSAMPLES,np.bincount(binidx,weights=values,minlength=NBINS**2)/np.maximum(counts,1),np.nan).reshape(NBINS,NBINS).T

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    if 'pr' not in ds:
        raise KeyError(f'IMERG `pr` is not in {SPLIT}.h5')
    ntime,nlat,nlon = (ds.sizes[dim] for dim in ['time','lat','lon'])
    lon   = ds['lon'].values
    hours = ds['time'].dt.hour.values
    era5,imerg,shf = (ds[name].transpose('time','lat','lon').values.ravel() for name in ['tp','pr','shf'])
    lf    = xr.broadcast(ds['lf'],ds['tp'])[0].transpose('time','lat','lon').values.ravel().astype(np.float64)
imerg      = imerg*TIMEWINDOW
thetaeanom = load_features(SPLIT,ATMRUN,CONFIG)[0]['thetae'].values*STATS['thetae_std']
preds      = {name:load_predictions(name) for name in LABELS}
preds      = {name:pred for name,pred in preds.items() if pred is not None}
seedmean   = {name:np.nanmean(pred,axis=0) for name,pred in preds.items()}
common     = np.isfinite(era5)&np.isfinite(imerg)&np.all([np.isfinite(pred) for pred in seedmean.values()],axis=0)
localhour  = (hours[:,None,None]+TIMEWINDOW/2+lon[None,None,:]/15.0)%24
hourbin    = (np.broadcast_to(localhour,(ntime,nlat,nlon)).ravel()//(24/NHOURBINS)).astype(int)
hourcenters = (np.arange(NHOURBINS)+0.5)*24/NHOURBINS
regions    = {'All':common,'Ocean':common&(lf<LANDFRAC),'Land':common&(lf>=LANDFRAC)}
print(f'{common.sum():,} samples with finite ERA5, IMERG, and predictions ({common.mean():.2%})')
print(f"Mean 3-hourly precipitation (mm): ERA5 {era5[common].mean():.3f}, IMERG {imerg[common].mean():.3f}")
print(f"R2 of ERA5 against IMERG: "+', '.join(f'{region} {calc_r2(era5,imerg,mask):.3f}' for region,mask in regions.items()))

In [ ]:
rows = []
for name,pred in preds.items():
    row = dict(model=LABELS[name])
    for target,true in [('ERA5',era5),('IMERG',imerg)]:
        for region,mask in regions.items():
            row[f'{target}_{region}'] = np.mean([calc_r2(seedpred,true,mask) for seedpred in pred])
    rows.append(row)
table = pd.DataFrame(rows).set_index('model').sort_values('ERA5_All')
pd.set_option('display.width',250)
print('Test R2 against ERA5 and IMERG:')
print(table.round(3).to_string())

In [ ]:
cycles,rows = {},[]
for region in ['Ocean','Land']:
    mask = regions[region]
    for source,values in [('ERA5',era5),('IMERG',imerg)]+[(name,seedmean[name]) for name in CYCLEMODELS]:
        cycles[(region,source)] = calc_diurnal_cycle(values,mask)
    for source in ['ERA5','IMERG']+CYCLEMODELS:
        cycle = cycles[(region,source)]
        rows.append(dict(region=region,source=LABELS.get(source,source),
                         meanratio_imerg=cycle.mean()/cycles[(region,'IMERG')].mean(),
                         amplitudepct_era5=100*np.ptp(cycle)/np.ptp(cycles[(region,'ERA5')]),
                         amplitudepct_imerg=100*np.ptp(cycle)/np.ptp(cycles[(region,'IMERG')]),
                         peakhour=hourcenters[np.argmax(cycle)]))
print('Diurnal cycles (mean relative to IMERG, amplitude as % of ERA5 and IMERG, peak LST):')
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
oceanmask = regions['Ocean']&np.isfinite(thetaeanom)&np.isfinite(shf)&(seedmean['sr_atm_eq']>=MINPRECIP)
strong    = oceanmask&(shf>=np.quantile(shf[oceanmask],TOPQUANTILE))&(thetaeanom>=np.quantile(thetaeanom[oceanmask],TOPQUANTILE))
print(f'Ocean samples in the top {100*(1-TOPQUANTILE):.0f}% of both SHF and kernel-integrated thetae (SR-ATM >= {MINPRECIP} mm): {strong.sum():,}')
for source,values in [('ERA5',era5),('IMERG',imerg),('SR-ATM',seedmean['sr_atm_eq']),('SR-ALL-PC',seedmean['sr_all_pc_eq'])]:
    print(f'   {source}: {values[strong].mean():.3f} mm')
edges   = (np.linspace(*np.percentile(shf[oceanmask],[1,99]),NBINS+1),np.linspace(*np.percentile(thetaeanom[oceanmask],[1,99]),NBINS+1))
centers = [0.5*(edge[:-1]+edge[1:]) for edge in edges]
needed  = {source:bin_mean(shf[oceanmask],thetaeanom[oceanmask],(values-seedmean['sr_atm_eq'])[oceanmask],edges) for source,values in [('ERA5',era5),('IMERG',imerg)]}
both    = np.isfinite(needed['ERA5'])&np.isfinite(needed['IMERG'])
print(f"Correlation of ERA5 - SR-ATM and IMERG - SR-ATM across SHF-thetae bins: {np.corrcoef(needed['ERA5'][both],needed['IMERG'][both])[0,1]:.3f}")

In [ ]:
handles = [Line2D([],[],color=COLORS[source],linewidth=2,label=source) for source in ['ERA5','IMERG']]
handles+= [Line2D([],[],color=COLORS[name],linewidth=2,label=LABELS[name]) for name in PLOTMODELS]
fig,axs = pplt.subplots([[1,2],[3,4]],refwidth=2.4,share=False,span=False,wspace=1.5,hspace=4)
for ax,region in zip(axs[:2],['Ocean','Land']):
    for source in ['ERA5','IMERG']+PLOTMODELS:
        ax.plot(hourcenters,cycles[(region,source)],color=COLORS[source],linewidth=2)
    ax.format(title=f'Diurnal Cycle ({region})',xlim=(0,24),xticks=[6,12,18],xticklabels=['06 LST','12 LST','18 LST'],ylim=(0,2),ylabel='Total Precipitation (mm)')
axs[1].legend(handles,loc='r',ncols=1,frame=False)
for ax,source in zip(axs[2:],['ERA5','IMERG']):
    m = ax.pcolormesh(*centers,needed[source],cmap='ColdHot_r',vmin=-4,vmax=4,levels=32,extend='both')
    ax.set_facecolor('gray1')
    ax.set_box_aspect(1)
    ax.format(title=f'{source} $-$ SR-ATM (Ocean)',xlabel='SHF (W/m$^2$)',ylabel='$\\widehat{\\mathit{\\theta_e}}-\\mathit{\\mu_{\\theta_e}}$ (K)',
              xlim=(edges[0][0],edges[0][-1]),ylim=(edges[1][0],edges[1][-1]))
fig.colorbar(m,loc='b',cols=(1,2),ticks=1,label='$\\Delta$Precipitation (mm)')
fig.format(abc=True,titleloc='l',grid=False)
pplt.show()
fig.save(FIGPATH)